# KAFKA INTERVIEW QUESTION ANSWERS

## PART 1:

 1. What is Apache Kafka?

Apache Kafka is an open-source distributed event streaming platform. It was built at LinkedIn and is now maintained by the Apache Software Foundation.Kafka lets one application publish a stream of events (things that happened, like "user clicked", "order placed", "payment failed") and lets many other applications read those events, in order, reliably, and at very high speed.

Three things make Kafka special:
- Fast (millions of messages per second)
- Durable (messages are written to disk and replicated, not just held in memory)
- Scalable (add more machines to handle more data)


2. Why is Kafka used in data engineering?

Imagine a company with 20 systems: website, mobile app, payments service, CRM, data warehouse, fraud engine, dashboards. If every system talks directly to every other system, you get a tangled web of point-to-point connections, painful to build and even more painful to maintain.

Kafka sits in the middle. Each system just talks to Kafka: producers write to it, consumers read from it.

- Decoupling: the producer doesn't care who reads the data or how many readers there are. Consumers can be added or removed without touching the producer.
- Real-time pipelines: data flows within milliseconds instead of waiting for a nightly batch job.
- Buffering / shock absorber: if a downstream database is slow or down, Kafka holds the data until it recovers.
- Replayability: data stays on Kafka for a configured time (e.g. 7 days). If you find a bug in your pipeline, fix it and re-read the old data.
- Scalability and fault tolerance: handles huge volumes and survives machine failures.
- Ecosystem: Kafka Connect (move data in and out of databases), Kafka Streams, ksqlDB, integrations with Spark, Flink, Snowflake, etc.

Typical use cases: clickstream ingestion, change data capture (CDC) from databases, log aggregation, IoT sensor data, fraud detection, feeding a data lake or warehouse in near real time.

3. What are the main components of Kafka?

- Producer: application that writes (publishes) messages to Kafka. Analogy: person posting letters.
- Consumer: application that reads (subscribes to) messages. Analogy: person collecting letters.
- Topic: named category/stream of messages. Analogy: a mail category like "Orders".
- Partition: a slice of a topic; an ordered, append-only log. Analogy: a separate shelf within that category.
- Broker: a Kafka server that stores partitions and serves clients. Analogy: one post-office building.
- Cluster: a group of brokers working together. Analogy: the whole postal network.
- Replica: a copy of a partition on another broker. Analogy: a photocopy kept in another building.
- Consumer group: set of consumers sharing the work of reading a topic. Analogy: a team of postmen splitting the route.
- Offset: sequential number of a message within a partition. Analogy: page number in a notebook.
- Controller (KRaft/ZooKeeper): manages cluster metadata and leader election. Analogy: the postal head office.

Note: older Kafka versions used Apache ZooKeeper to manage cluster metadata. Modern Kafka uses its own built-in consensus layer called KRaft, and ZooKeeper is no longer needed.

## PART 2: TOPICS, PARTITIONS, BROKERS AND CLUSTERS


### 4. What is a Kafka topic?

A topic is a named stream of messages. Producers write to a topic and consumers read from a topic. Think of it like a table name in a database or a folder name in a file system, except it is an append-only log: new messages are added at the end and existing ones are not modified.

Examples: orders, user-clicks, payments.

Messages in a topic are kept for a retention period (time-based, like 7 days, or size-based), not deleted when someone reads them. That is a big difference from classic queues.

### 5. What is a partition in Kafka?

A topic is too big to live in one place, so Kafka splits it into partitions. Each partition is an ordered, immutable sequence of messages stored as a log on disk. Every message in a partition gets a sequential number called an offset (0, 1, 2, 3, ...).

```
Topic: orders (3 partitions)
Partition 0: [0][1][2][3][4][5] -> new messages appended here
Partition 1: [0][1][2][3]       ->
Partition 2: [0][1][2][3][4]    ->
```

Ordering is guaranteed only inside one partition, not across the whole topic.

### 6. Why does Kafka use partitions?

- Scalability (storage): one machine may not hold all of a huge topic. Partitions can live on different brokers.
- Parallelism (write): several producers can write to different partitions at the same time.
- Parallelism (read): in a consumer group, each partition can be read by a different consumer, so reading speed scales with the number of partitions.
- Load distribution: work is spread across many brokers instead of overloading one.
- Ordering with scale: by sending related messages (same key) to the same partition, you keep ordering where it matters while still scaling out.

Analogy: one checkout counter at a supermarket makes a long queue. Opening five counters (partitions) lets five cashiers (consumers) work in parallel.

### 7. What is a broker?

A broker is a single Kafka server (a running Kafka process on a machine or container). It receives messages from producers, stores them on disk as partition logs, and serves them to consumers. Each broker has a unique numeric ID (e.g. 1, 2, 3). A broker does not hold everything; it holds only some partitions of some topics.

### 8. What is a Kafka cluster?

A cluster is a group of brokers working together as one system. Typical production clusters have 3 or more brokers. The partitions of a topic are spread across the brokers, and copies (replicas) are placed on different brokers for safety. Clients only need to know one or two broker addresses (bootstrap servers); they discover the rest of the cluster automatically.

```
Cluster
 +-- Broker 1: orders-P0 (leader), orders-P1 (follower)
 +-- Broker 2: orders-P1 (leader), orders-P2 (follower)
 +-- Broker 3: orders-P2 (leader), orders-P0 (follower)
```

### 9. What is the difference between a topic and a partition?

Topic:
- Logical name/category for a stream of messages
- What producers and consumers refer to by name
- Has no ordering guarantee across the whole topic
- Made up of one or more partitions
- Example: orders

Partition:
- Physical slice of a topic where data is actually stored
- What Kafka uses internally for storage, replication and parallelism
- Strict ordering of messages inside it
- Belongs to exactly one topic
- Example: orders-0, orders-1, orders-2

Analogy: Topic = a book. Partitions = the chapters stored in different cupboards. You talk about the book, but the content physically lives in the chapters.

### 10. Can a topic have multiple partitions?

Yes, and in real projects almost every topic does. You choose the number when creating it:

```
kafka-topics.sh --create --topic orders --partitions 3 --replication-factor 3 --bootstrap-server localhost:9092
```

A topic with a single partition is possible but has no parallelism (only one consumer in a group can read it) and cannot scale.

### 11. Can multiple consumers read from the same topic?

Yes. Kafka does not delete messages when they are read, so there is no "first come, first served" fight over messages. There are two ways multiple consumers can read the same topic:

- Same consumer group: they share the work. Each partition goes to only one consumer in the group, so each message is processed once by the group.
- Different consumer groups: each group gets its own full copy of the stream. For example, one group feeds the data warehouse while another feeds a fraud-detection service.

### 12. What happens when you increase the number of partitions of a topic?

You can increase partitions (never decrease them). Here is what changes:

- More parallelism: more consumers in a group can now be active.
- Key-to-partition mapping changes: the default partitioner uses hash(key) % number_of_partitions. When the partition count changes, the same key may map to a different partition than before. So messages with key "customer-42" that used to go to partition 1 might now go to partition 4.
- Ordering risk: old messages for that key are in the old partition and new ones in the new partition, so strict per-key ordering across the change is broken.
- Existing data is NOT moved: Kafka doesn't re-shuffle old messages. Only new messages use the new layout.
- Rebalance: consumer groups rebalance to pick up the new partitions.
- Cannot be reduced: to shrink, you need to create a new topic and migrate the data.

Advice: plan the partition count up front, leave some headroom for growth, and be careful when you rely on key-based ordering.

```
kafka-topics.sh --alter --topic orders --partitions 6 --bootstrap-server localhost:9092
```

## PART 3: REPLICATION AND FAULT TOLERANCE


### 13. What is replication in Kafka?

Replication means Kafka keeps multiple copies of every partition on different brokers. One copy is the leader (handles all reads and writes for that partition) and the others are followers (they quietly copy everything from the leader). If the leader's broker dies, one of the followers is promoted to be the new leader and life goes on.

### 14. What is a replication factor?

The replication factor is how many total copies of each partition exist, including the leader. Replication factor = 3 means 1 leader + 2 followers. The common production setting is 3.

```
Topic orders, partition 0, replication factor 3:
 Broker 1: Leader
 Broker 2: Follower
 Broker 3: Follower
```

The replication factor cannot be larger than the number of brokers (you can't keep 3 copies on a 2-broker cluster, because two copies on the same broker would protect nothing).

### 15. Why is replication important?

- No data loss if a disk or machine fails.
- High availability: the cluster keeps working when a broker goes down.
- Durability guarantees: combined with producer setting acks=all and broker setting min.insync.replicas, a message is confirmed only after it is safely on multiple brokers.

Without replication, one broken server could mean permanently lost data and an outage.

### 16. What happens if a Kafka broker goes down?

Step by step:
- The cluster controller notices the broker has stopped sending heartbeats.
- For every partition whose leader was on that broker, the controller picks a new leader from the in-sync replicas (ISR), which are followers that are fully caught up.
- Producers and consumers get updated metadata and automatically switch to the new leader. Usually this takes only seconds, and sometimes a short pause or retry is visible.
- Partitions where the dead broker only held a follower copy are unaffected, though they now have one fewer copy (under-replicated).
- When the broker comes back, it rejoins, catches up from the current leaders, and rejoins the ISR.

When is data actually lost? Only if all replicas of a partition are gone, or if an out-of-sync replica is forced to become leader (unclean leader election, disabled by default). That is why replication factor 3 is recommended.

In short: Followers take over as leader; the application continues with minimal disruption.

### 17. What are follower replicas?

Follower replicas are the backup copies of a partition. They constantly send fetch requests to the leader and append the same messages in the same order. Normally clients do not read from or write to followers (writes always go to the leader; newer Kafka versions allow consumers to optionally fetch from a nearby follower to save cross-zone network cost).

A follower that keeps up with the leader is called in-sync; the set of all in-sync replicas is the ISR. A follower that falls too far behind is removed from the ISR and can't become leader until it catches up.

### 18. What is the difference between Kafka and a traditional message queue?

(Traditional queue = RabbitMQ, ActiveMQ)

Message after reading:
- Queue: usually deleted once consumed and acknowledged
- Kafka: stays until retention expires; any number of readers can read it

Model:
- Queue: broker pushes messages to consumers
- Kafka: consumers pull at their own pace

Replay:
- Queue: generally not possible
- Kafka: easy, just reset the offset

Ordering:
- Queue: per queue
- Kafka: per partition

Scaling:
- Queue: mostly vertical or via competing consumers
- Kafka: horizontal, by partitions and brokers

Throughput:
- Queue: thousands to tens of thousands per second
- Kafka: hundreds of thousands to millions per second

Multiple independent subscribers:
- Queue: needs extra setup (exchanges, fan-out queues)
- Kafka: built in through consumer groups

Storage role:
- Queue: temporary buffer
- Kafka: durable log; can act as a source of truth

Best for:
- Queue: task queues, complex routing, request/reply
- Kafka: event streaming, data pipelines, analytics

Analogy: a queue is like a to-do tray: when you take a task, it is gone. Kafka is like a newspaper archive or a security-camera recording: everyone can watch, each person has their own bookmark, and you can rewind.

## PART 4: PRODUCERS


### 19. What is a Kafka producer?

A producer is any application or service that writes messages to a Kafka topic. It can be a web server logging clicks, a microservice publishing "order created" events, or a CDC tool like Debezium streaming database changes. Producers are written using client libraries (Java, Python, Go, etc.).

### 20. How does a producer send data to Kafka?

1. Create a record: topic + optional key + value (+ optional headers and timestamp).
2. Serialize: key and value are converted into bytes (String, JSON, Avro, Protobuf...).
3. Partitioner: decides which partition the record goes to.
4. Buffer and batch: the record is placed into an in-memory batch for that partition.
5. Sender thread: sends the batch to the leader broker of that partition.
6. Leader writes to its log, followers replicate it.
7. Acknowledgement: depending on acks, the broker confirms back to the producer. On failure the producer can retry.

```
from kafka import KafkaProducer
producer = KafkaProducer(bootstrap_servers="localhost:9092", acks="all")
producer.send("orders", key=b"customer-42", value=b'{"order_id": 101}')
producer.flush()
```

acks setting:
- acks=0: don't wait for any confirmation. Fastest, may lose data.
- acks=1: wait for the leader only. Balanced; loses data if the leader dies before followers copy.
- acks=all (-1): wait for all in-sync replicas. Safest, slightly slower.

### 21. How does a producer decide which partition receives a message?

The producer's partitioner decides, using this order of rules:
1. If the record explicitly specifies a partition number, use that.
2. Else, if the record has a key, compute hash(key) % number_of_partitions (murmur2 hash in the default Java client). Same key gives the same partition.
3. Else (no key), spread the records across partitions. Older clients used round-robin; newer ones use a sticky partitioner that fills a batch for one partition before moving to another, which is more efficient.
4. You can also plug in a custom partitioner with your own business logic.

### 22. What is a partition key?

A partition key is an optional value attached to a message (e.g. customer_id, user_id, device_id) that is used to choose the partition. It isn't the same as a database primary key: it doesn't need to be unique. Many messages can share one key, and that is the whole point.

### 23. Why is a key important when producing Kafka messages?

- Ordering: all events for the same key go to the same partition, so they are read in the order they were written. Example: "account created", "deposit", "withdrawal" for account 42 must be processed in this order.
- Co-location: all data of one entity lives together, which makes stateful processing (aggregations, joins, deduplication) easy and correct.
- Log compaction: on compacted topics Kafka keeps only the latest value per key, useful for "current state" topics.
- Even distribution: a good key with many distinct values spreads load. A bad key (e.g. country = "IN" for 90% of data) creates a hot partition.

### 24. What happens if the producer doesn't provide a key?

The key is null, so Kafka spreads the messages across partitions (round-robin or sticky batching). You get good load balancing but no ordering guarantee across messages that logically belong together, because they may land in different partitions. That is perfectly fine for things like independent log lines or metrics, and not fine for things like a bank account's transactions.

### 25. If you send records with the same key, will they always go to the same partition?

Almost always, but not unconditionally. The honest answer has caveats:

- Yes, as long as the number of partitions in the topic stays the same and the same partitioner logic is used. The hash is deterministic.
- No, if you increase the partition count. The modulo changes, so the key can map to a different partition afterwards.
- No, if different producers use different partitioner algorithms (e.g. a Java client and a librdkafka client with different hash functions) or if a custom partitioner is used.
- No, if a partition number is set explicitly, which overrides the key.

In short: Same key goes to the same partition, provided the partition count and partitioner don't change.

### 26. What is producer batching?

Instead of sending each message over the network separately, the producer collects several messages into a batch and sends them together in one request. It is like a delivery van waiting to be reasonably full instead of driving out with one parcel at a time.

- batch.size: maximum size (bytes) of a batch per partition (default 16 KB).
- linger.ms: how long the producer waits for more messages before sending a not-yet-full batch (default 0).
- compression.type: batches can be compressed (snappy, lz4, zstd, gzip), and compression works better on bigger batches.
- buffer.memory: total memory the producer may use for buffering.

Benefits: far higher throughput, fewer network calls, better compression, less broker load.  
Trade-off: a bigger linger.ms adds a little latency. Tune it for your needs (e.g. 5 to 20 ms for throughput-heavy pipelines).

## PART 5: CONSUMERS AND OFFSETS


### 27. What is a Kafka consumer?

A consumer is an application that subscribes to one or more topics and reads messages from them. Examples: a Spark job loading data to a warehouse, a fraud service, a microservice reacting to "order placed" events.

### 28. How does a consumer read messages from Kafka?

Kafka uses a pull model: the consumer asks the broker for data; the broker does not push. The loop is:

1. Subscribe to a topic (and usually join a consumer group).
2. Poll: call poll() in a loop. The broker returns a batch of messages starting from the consumer's current offset for each assigned partition.
3. Deserialize the bytes back into objects.
4. Process the messages (write to DB, transform, call API...).
5. Commit offsets to record progress (automatically or manually).

```
from kafka import KafkaConsumer
consumer = KafkaConsumer("orders",
    bootstrap_servers="localhost:9092",
    group_id="billing-service",
    auto_offset_reset="earliest",
    enable_auto_commit=False)
for msg in consumer:
    process(msg.value)
    consumer.commit()
```

Pulling lets each consumer read at its own speed. A slow consumer simply falls behind (this is called lag) without hurting anyone else.

### 29. What is an offset?

An offset is the unique, sequential position number of a message inside a partition, starting from 0. It is like a page number or a line number in a log file. Offsets are per partition, so partition 0 offset 5 and partition 1 offset 5 are completely different messages.

Two related ideas:
- Message offset: where a message sits in the log.
- Committed offset: the next position a consumer group should read from after a restart, stored in an internal topic called __consumer_offsets.

### 30. Why are offsets important?

- Track progress: they tell a consumer group exactly where it stopped.
- Resume after a crash: on restart, consumption continues from the last committed offset instead of starting over.
- Delivery guarantees: when you commit decides the semantics.
    - Commit before processing = at-most-once (risk of loss).
    - Commit after processing = at-least-once (risk of duplicates).
    - Transactions or idempotent writes can achieve exactly-once.
- Replay: you can rewind offsets to reprocess data after a bug fix.
- Monitoring: consumer lag = latest offset minus committed offset, the key health metric for pipelines.

### 31. What is auto.offset.reset?

It is a consumer setting that tells Kafka what to do when there is no valid committed offset for the group and partition. That happens when:  
(a) a brand-new consumer group starts for the first time, or  
(b) the stored offset no longer exists, e.g. because that data was deleted by retention.

Important: it does not apply when the group already has a committed offset. In that case the consumer simply resumes from it.

Values:
- earliest: start from the oldest message still available in the partition
- latest (default): start from the end; only read messages that arrive after the consumer connects
- none: throw an error if no committed offset is found

### 31a. Explain: earliest

earliest means "give me everything that is still stored, from the beginning."

Example: a partition holds offsets 0 to 999 (old messages not yet removed by retention). A new consumer group starts with earliest. It starts at offset 0 and processes all 1000 existing messages, then continues with new ones.

Use when: you must not miss any data, such as loading a data warehouse, back-filling, reprocessing history, or testing.  
Watch out: the first run may be heavy because it processes all retained history.

### 31b. Explain: latest

latest means "ignore history; I only care from now on."

Example: same partition with offsets 0 to 999. A new group starts with latest. It jumps to offset 1000 (the end) and only sees messages produced after it joined. The earlier 1000 are skipped.

Use when: you only need live data, such as real-time dashboards or alerting.  
Watch out: anything produced before your consumer first started is missed. A common beginner surprise: "I started my consumer and nothing shows up!", because the old messages are skipped.

```
Partition: [0][1][2] ... [997][998][999] | (next message will be 1000)
earliest -> starts at offset 0 (reads all 1000 old + new)
latest   -> starts at offset 1000 (reads only new)
```

## PART 6: CONSUMER GROUPS IN DEPTH


### 32. What is a consumer group?

A consumer group is a set of consumers that share the same group.id and cooperate to read a topic. Kafka divides the topic's partitions among the members of the group. Each partition is read by exactly one consumer in that group at a time, so together the group processes each message once.

Analogy: a team of 3 people sorting 6 bags of mail; each person takes 2 bags. Nobody touches someone else's bag, and all mail gets sorted.

### 33. Why do we use consumer groups?

- Scale out processing: add more consumers to read faster.
- No duplicate work: members split partitions, so a message isn't processed twice by the same group.
- Fault tolerance: if one member dies, its partitions go to the others.
- Independent applications: different teams/apps use different groups and each gets the full data stream.
- Offset tracking per group: each group remembers its own position.

### 34. Can two consumers from the same consumer group consume the same partition simultaneously?

No. Within a consumer group, a partition is assigned to only one consumer at a time. This rule is what guarantees in-order processing within a partition and prevents duplicate processing inside the group. (It can move to another consumer later, after a rebalance, but never to two at once.)

### 35. Can two consumers from different consumer groups consume the same message?

Yes, absolutely. Each group has independent offsets, so each group reads every message on its own. This is Kafka's publish/subscribe superpower.

```
Topic: orders
 Group "billing":   reads all orders -> creates invoices
 Group "analytics": reads all orders -> loads warehouse
 Group "fraud":     reads all orders -> checks suspicious activity
(each group gets every message; none of them interferes with the others)
```

### 36. What happens if a consumer crashes?

- The consumer stops sending heartbeats to the group coordinator (a broker).
- After session.timeout.ms (or if it stops polling for longer than max.poll.interval.ms), the coordinator declares it dead.
- A rebalance is triggered: the dead consumer's partitions are reassigned to the surviving members.
- The new owners resume from the last committed offset of those partitions.

Consequence: any messages processed but not yet committed by the crashed consumer are processed again (duplicates, so design consumers to be idempotent). Messages are not lost, because they're still in Kafka. That is at-least-once delivery.


### 37. What happens when a new consumer joins a consumer group?

- The new consumer contacts the group coordinator and sends a JoinGroup request.
- A rebalance starts. Partition assignments are recalculated across all members, including the newcomer.
- Some partitions are taken from existing consumers and given to the new one.
- Each consumer continues from the committed offset of its partitions.

Example: 6 partitions, 2 consumers (3 each). A third consumer joins. After rebalance: 2 partitions each.

### 38. What happens when the number of consumers is greater than the number of partitions?

Because one partition can only be read by one consumer in a group, the extra consumers sit idle. They are connected to the group and healthy, but have no partitions assigned and receive no data.

Example: topic with 3 partitions, group with 5 consumers:
```
Partition 0 -> Consumer A
Partition 1 -> Consumer B
Partition 2 -> Consumer C
Consumer D -> idle
Consumer E -> idle
```

Are the idle ones useless? Not completely. They act as hot standbys: if A, B or C crashes, an idle consumer takes over quickly. But for throughput, they add nothing.

Lesson: the maximum useful parallelism of a consumer group equals the number of partitions. If you need more parallelism, increase the partitions (carefully, see question 12).

39. What happens when the number of consumers is less than the number of partitions?

This is very normal and works perfectly. Some consumers simply get more than one partition. Kafka spreads the partitions as evenly as it can.

Example 1: 6 partitions, 3 consumers:
```
Consumer A -> P0, P1
Consumer B -> P2, P3
Consumer C -> P4, P5
```

Example 2 (uneven): 5 partitions, 2 consumers. Assignment may be 3 and 2:
```
Consumer A -> P0, P1, P2
Consumer B -> P3, P4
```

Example 3: only 1 consumer with 4 partitions. That single consumer reads all four partitions.

- Every message is still processed, and nothing is skipped.
- Ordering within each partition is preserved. A consumer with several partitions may interleave messages from them, but each partition's order stays intact.
- The load may be uneven (one consumer handles 3 partitions while the other handles 2), and consumers with more partitions may fall behind, increasing lag.
- If lag keeps growing, add consumers (up to the number of partitions) and they will share the load after a rebalance.